# Precipitación Corn Belt, NOAA NCEI

Descarga precipitación por estado para los 8 estados del Corn Belt:

1. **Mensual**: Climate at a Glance (nClimDiv), pulgadas convertidas a mm.
2. **Diaria**: nClimGrid-Daily v1.0.0, promedios areales por estado, en mm.

Al final compara la suma de los diarios contra el mensual oficial y permite descargar los CSV.

## 1. Parámetros
Cambiá `INICIO`, `FIN` y `ESTADOS` según lo que necesites. Los códigos son los de NCEI.

In [ ]:
from datetime import date

import pandas as pd
import requests

INICIO = (2021, 9)                          # (año, mes)
FIN = (date.today().year, date.today().month)

# Nombre del estado -> código NCEI (Climate at a Glance)
ESTADOS = {
    "Illinois": 11,
    "Indiana": 12,
    "Iowa": 13,
    "Minnesota": 21,
    "Missouri": 23,
    "Nebraska": 25,
    "Ohio": 33,
    "South Dakota": 39,
}

URL_CAG = ("https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/"
           "statewide/time-series/{code}/pcp/1/0/{y0}-{y1}/data.json")
URL_DIARIA = ("https://www.ncei.noaa.gov/data/nclimgrid-daily/access/averages/"
              "{y}/prcp-{y}{m:02d}-ste-{version}.csv")

session = requests.Session()
session.headers["User-Agent"] = "precip-cornbelt (uso academico)"


def meses(inicio, fin):
    y, m = inicio
    while (y, m) <= fin:
        yield y, m
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)

## 2. Precipitación mensual (Climate at a Glance)

In [ ]:
def descargar_mensual():
    filas = []
    for estado, code in ESTADOS.items():
        url = URL_CAG.format(code=code, y0=INICIO[0], y1=FIN[0])
        data = session.get(url, timeout=60).json()["data"]
        for yyyymm, d in data.items():
            y, m = int(yyyymm[:4]), int(yyyymm[4:])
            if INICIO <= (y, m) <= FIN:
                filas.append({
                    "estado": estado,
                    "fecha": f"{y}-{m:02d}-01",
                    "precip_mm": round(float(d["value"]) * 25.4, 2),
                })
        print(f"mensual  {estado}: ok")
    return pd.DataFrame(filas)


mensual = descargar_mensual()
mensual.to_csv("precip_mensual.csv", index=False)
mensual.head()

## 3. Precipitación diaria (nClimGrid-Daily)
Un CSV por mes con todos los estados. Cada fila: tipo, código, nombre, año, mes, elemento, día 1 a 31. El valor -999.99 marca días inexistentes o sin dato. Si la versión final (`scaled`) no está, usa la preliminar.

In [ ]:
def leer_mes(y, m):
    for version in ("scaled", "prelim"):
        r = session.get(URL_DIARIA.format(y=y, m=m, version=version), timeout=60)
        if r.ok:
            return r.text, version
    return None, None


def descargar_diaria():
    filas = []
    for y, m in meses(INICIO, FIN):
        texto, version = leer_mes(y, m)
        if texto is None:
            print(f"diaria   {y}-{m:02d}: no disponible")
            continue
        for linea in texto.splitlines():
            p = [x.strip() for x in linea.split(",")]
            if len(p) < 7 or p[2] not in ESTADOS:
                continue
            for dia, valor in enumerate(p[6:], start=1):
                v = float(valor)
                if v <= -999:
                    continue
                filas.append({
                    "estado": p[2],
                    "fecha": f"{y}-{m:02d}-{dia:02d}",
                    "precip_mm": v,
                    "calidad": "final" if version == "scaled" else "preliminar",
                })
        print(f"diaria   {y}-{m:02d}: ok ({version})")
    return pd.DataFrame(filas)


diaria = descargar_diaria()
diaria.to_csv("precip_diaria.csv", index=False)
print(diaria.shape)
diaria.head()

## 4. Control: suma de diarios vs mensual oficial
Muestra los meses con diferencia mayor a 5 mm (julio 2023 debería aparecer, es una inconsistencia del archivo de NOAA).

In [ ]:
diaria["mes"] = diaria["fecha"].str[:7] + "-01"
control = (diaria.groupby(["estado", "mes"])["precip_mm"].sum()
           .rename("suma_diaria").reset_index()
           .merge(mensual.rename(columns={"fecha": "mes", "precip_mm": "mensual"}),
                  on=["estado", "mes"]))
control["dif_mm"] = (control["suma_diaria"] - control["mensual"]).round(1)
control[control["dif_mm"].abs() > 5]

## 5. Descargar los CSV

In [ ]:
from google.colab import files
files.download("precip_mensual.csv")
files.download("precip_diaria.csv")